In [2]:
# CPSC 3720 Assignment 2
# Analyzing AESO Electricity Demand and Price

# 1. Import Libraries
import pandas as pd
import matplotlib.pyplot as plt

print("Starting analysis...")

Starting analysis...


In [ ]:
# --- 2. Data Loading and Cleaning ---

# Load the dataset downloaded from AESO
# AESO CSVs often have several header rows to skip. 
# We'll inspect the file and skip them.
# 'header=3' usually works for this specific report.
try:
    df = pd.read_csv('aeso_data.csv', header=0)
except FileNotFoundError:
    print("Error: 'aeso_data.csv' not found.")
    print("Please download the file from the AESO website and save it in the correct folder.")
    exit()

# Rename columns for easier access.
# 'Alberta Internal Load (AIL)' is the demand.
df.rename(columns={
    'Date_Begin_GMT': 'datetime',
    'ACTUAL_AIL': 'demand_mw',
    'ACTUAL_POOL_PRICE': 'price_cad'
}, inplace=True)


# Create a proper datetime column.
# The 'hour' column is 1-24. We'll subtract 1 to make it 0-23 for proper time parsing.
# df['hour'] = df['hour'].apply(lambda x: str(int(x) - 1))
# df['datetime_str'] = df['date'] + ' ' + df['hour'] + ':00'

# Convert to datetime objects. This is crucial for time series analysis.
# df['datetime'] = pd.to_datetime(df['datetime_str'], format='%Y-%m-%d %H:%M')

# Set the datetime as the index
df.set_index('datetime', inplace=True)

# Select only the columns we need
df_cleaned = df[['demand_mw', 'price_cad']].copy()

# Ensure data is numeric (it might load as string '1,000')
df_cleaned['demand_mw'] = pd.to_numeric(df_cleaned['demand_mw'].astype(str).str.replace(',', ''))
df_cleaned['price_cad'] = pd.to_numeric(df_cleaned['price_cad'].astype(str).str.replace(',', ''))

# For this example, let's just use data from 2023 for a clearer plot
df_2023 = df_cleaned.loc['2023-01-01 0:00':'2023-12-31 23:00']

print(f"Data loaded and cleaned. Total rows: {len(df_2023)}")
print(df_2023)
# --- 3. Data Manipulation ---
# Simple plotting is not enough. We will calculate a 7-day running average
# to smooth out volatility and show longer-term trends.
# 24 hours * 7 days = 168
window_size = 24 * 7 

df_2023['demand_7day_avg'] = df_2023['demand_mw'].rolling(window=window_size).mean()
df_2023['price_7day_avg'] = df_2023['price_cad'].rolling(window=window_size).mean()

# --- 4. Visualization ---
# Create a figure with two subplots, sharing the x-axis
fig, (ax1, ax2) = plt.subplots(
    nrows=2, 
    ncols=1, 
    figsize=(15, 10), 
    sharex=True
)

# --- Subplot 1: Electricity Demand ---
ax1.set_title('Alberta Electricity Demand (AIL) - 2023', fontsize=16)

# Plot raw hourly demand (Series 1)
ax1.plot(df_2023.index, df_2023['demand_mw'], label='Hourly Demand (MW)', color='lightblue', alpha=0.8)

# Plot 7-day running average (Series 2)
ax1.plot(df_2023.index, df_2023['demand_7day_avg'], label='7-Day Running Average', color='blue', linewidth=2)

ax1.set_ylabel('Demand (MW)')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.5)

# --- Subplot 2: Electricity Price ---
ax2.set_title('Alberta Electricity Pool Price - 2023', fontsize=16)

# Plot raw hourly price (Series 3)
ax2.plot(df_2023.index, df_2023['price_cad'], label='Hourly Pool Price ($)', color='lightgreen', alpha=0.8)

# Plot 7-day running average (Series 4)
ax2.plot(df_2023.index, df_2023['price_7day_avg'], label='7-Day Running Average', color='green', linewidth=2)

ax2.set_ylabel('Price ($/MWh)')
ax2.set_xlabel('Date')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.5)

# Set a y-limit for price to cut off extreme spikes and make the trend visible
# You can comment this out to see the full spikes
ax2.set_ylim(-50, 300) 

# --- 5. Save and Show the Final Graphic ---
plt.suptitle('CPSC 3720 - Alberta Electricity Demand and Price Analysis (2023)', fontsize=20, y=1.03)
plt.tight_layout()

# Save the final image
plt.savefig('CPSC3720_Assignment2_AESO_Analysis.png')

print("Analysis complete. Plot saved as 'CPSC3720_Assignment2_AESO_Analysis.png'")

# Show the plot
plt.show()

# --- 6. Data Source Citation ---
# Data sourced from AESO, "Hourly Generation Metered Volumes and Pool Price and AIL data 2001 to July 2025".
# Retrieved from: https://www.aeso.ca/market/market-and-system-reporting/data-requests/hourly-generation-metered-volumes-and-pool-price-and-ail-data-2001-to-july-2025

                  Date_Begin_Local     AFG1    AKE1    ALP1    ALP2    APS1  \
datetime                                                                      
2020-01-01 7:00    2020-01-01 0:00  15.9740  69.105  0.0000  0.0000  0.0564   
2020-01-01 8:00    2020-01-01 1:00  15.9810  58.543  0.0000  0.0000  0.7626   
2020-01-01 9:00    2020-01-01 2:00  15.9830  66.649  0.0000  0.0000  2.1692   
2020-01-01 10:00   2020-01-01 3:00  15.9690  66.416  0.0000  0.0000  3.3344   
2020-01-01 11:00   2020-01-01 4:00  15.9750  54.956  0.0000  0.0000  1.1939   
...                            ...      ...     ...     ...     ...     ...   
2025-08-01 1:00   2025-07-31 19:00  16.8666   2.913  1.2846  1.8581  0.0000   
2025-08-01 2:00   2025-07-31 20:00  16.0451   2.171  0.0000  0.0000  0.2417   
2025-08-01 3:00   2025-07-31 21:00  16.1691   6.159  0.0000  0.0000  0.0000   
2025-08-01 4:00   2025-07-31 22:00  15.7849  11.116  0.0000  0.0000  0.0000   
2025-08-01 5:00   2025-07-31 23:00  16.8859  11.306 

C:\Users\adria\AppData\Local\Temp\ipykernel_29136\1229497500.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2023['demand_7day_avg'] = df_2023['demand_mw'].rolling(window=window_size).mean()
C:\Users\adria\AppData\Local\Temp\ipykernel_29136\1229497500.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2023['price_7day_avg'] = df_2023['price_cad'].rolling(window=window_size).mean()


Analysis complete. Plot saved as 'CPSC3720_Assignment2_AESO_Analysis.png'


KeyboardInterrupt: 